#Langchain?

open-source framework used to develop the applications powered by llm's
it helps in the connection of LLM's with the external data sources - document, API's, tool
it helps in the simplification in building the bots - Q&A bot, RAG applications.

#Compoents of Langchain?
Model -> laguage models that understand prompts (users)


2) Prompts -> questions / queries that u asks from the bot

3) Output parsers -> covert model output/responses into a structured format.
ex -> List, JSON
4) Document loaders -> loads data from the exteral sources.. PDF, text files(.txt)....
5) Text Splitters -> Divide large document into smaller chunks
6) Embeddings -> Convert these text into numerical vector
7) vector store -> store and search vector embeddigs.
8) Retievers -> retrieve relevat documents or text chunks based upon the query a user is askig
9) Chains (|) -> connect multiple steps together in order to complete the tasks.
10) Tools and Agents -> allow an LLMS to use the exteral tools.


example ->
RAG-Based College Chatbot

student -> What is the attendace requirement for the final examination?

step1 -> load the college data (rule policy) PDF.

step2 -> split the pdf content into chunks
step3 -> generate the embeddings

step4 -> store the embedding in the vector DB

step5 -> Retrieve the chunks containing the attendance rules

step6 -> pass the retrieved context and question to LLM          -> frame

step 7 -> generate the answeres based on the policy..

In [7]:
!pip install -qU langchain-google-genai langchain-community langchain chromadb

#-q -> quite (it reduce the amount of installation information displayed.)
#-u -> upgrade (it allows the pip to download or to fetch the latest version of all the packages)

# langchain-google-genai -> asking the question from gemini


In [8]:
from langchain_community.document_loaders import TextLoader                   #import the pdf or any doc file
from langchain_text_splitters import RecursiveCharacterTextSplitter           # splittig the etire data into small chuks
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI   #generate the embeddings for the input words
from langchain_community.vectorstores import Chroma                                 #vector database in which your entire seq is stored
from langchain_core.prompts import ChatPromptTemplate                          #help us to create a prompt template
from langchain_core.output_parsers import StrOutputParser                # to remove the extra iformation from the output
from langchain_core.runnables import RunnablePassthrough                 #create a runnable chain to parse through the entire sequence


In [9]:
import os
import getpass
import sys


In [10]:
GOOGLE_API_KEY  = "AQ.Ab8RN6IY66Gl3U2O1nbSYGaqUgnxeIBqg-2H4U-jNUTCH9_uFg"
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

In [11]:

def setup_env():
  if not os.getenv("GOOGLE_API_KEY"):
    os.envion["GOOGLE_API_KEY"]



In [12]:
def load_and_split(filepath):
  if not os.path.exists(filepath):
    print(f'Error: File not found at {filepath}')
    sys.exit(1)

  print(f'Loading Data')
  loader = TextLoader(filepath)      #for pdf
  docs = loader.load()


  print(f'Splitting the data into chunks')
  splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 200) #to define any connection in between the chunks
  splits = splitter.split_documents(docs)
  print(f'Split {len(splits)} chunks')
  return splits


In [13]:
#creating the vector db and rag chain

def create_rag_chain(splits):
  print(f'Initalize the vector size and Create a Rag Chain')
  embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-001",task_type = "retrieval_document")
  vectorstore = Chroma.from_documents(documents = splits, embedding =embeddings)
  retriever = vectorstore.as_retriever()    #this is my context

  llm = ChatGoogleGenerativeAI(model ="gemini-3.5-flash", temperature = 0)

  template = """Answer the Question based on the following context: {context}
    Question: {question}

    Helpful Answer: """

  prompt = ChatPromptTemplate.from_template(template)


  #create Doc with the content pages
  def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    #extract the text from each of the document using doc.page_content
    #join te text together with the two newline characters in between the documents


  #RAG CHAIN
  chain = (
      {"context" : retriever | format_docs, 'question':RunnablePassthrough()}
      | prompt
      | llm
      | StrOutputParser()
  )
  return chain


In [14]:
setup_env()
file_path = 'harrypotter 1.txt'
splits = load_and_split(file_path)
rag_chain = create_rag_chain(splits)


Loading Data
Splitting the data into chunks
Split 15 chunks
Initalize the vector size and Create a Rag Chain


In [15]:
message = input("Ask Question: ")
output = rag_chain.invoke(message)
print(output)


Ask Question:  how the thunder sign in his forehead related to voldamon
Based on the provided context, the lightning-shaped scar on Harry's forehead was left after he mysteriously survived Voldemort's failed killing curse as a baby.


In [16]:
message = input("Ask Question: ")
output = rag_chain.invoke(message)
print(output)


Ask Question: how voldmart and harry was linked
Based on the provided context, Harry and Voldemort were linked in the following ways:

* **As a Horcrux:** Harry himself was one of Voldemort's Horcruxes, meaning a piece of Voldemort's soul was inside him.
* **Through Fate/Prophecy:** Voldemort must kill Harry or vice versa.
* **Through Magic and Blood:** They were connected by magical technicalities involving the protection from Harry's mother and Voldemort's blood.
* **Through the Rebounded Curse:** Voldemort marked Harry as "The Boy Who Lived" after a failed curse against him rebounded.


In [17]:
message = input("Ask Question: ")
output = rag_chain.invoke(message)
print(output)


Ask Question: why harry is to be killed to kill voldmart
Based on the provided context, Harry must be killed/sacrifice himself because he learns that **he himself is a Horcrux** (containing a piece of Voldemort's split soul).
